# Ohùn: train a Yoruba voice on a free Kaggle GPU

**Settings:** Accelerator **GPU T4 x2** (training uses 1 GPU), Internet **On**, Kaggle secret **HF_TOKEN** (write) attached.

Data: OpenSLR 86 Yoruba (CC BY-SA 4.0). Base: Piper `ljspeech/medium`.

**Resuming:** the notebook looks for a previous `last.ckpt` in (1) any attached input (e.g. this notebook's earlier output), then (2) `checkpoints/last.ckpt` in the HF repo `aladewealth/ohun-yoruba`. If one is found it **exports and publishes it first** (so the API gets a voice within ~20 min), then keeps training. Every session ends by exporting, publishing the ONNX, and uploading `last.ckpt` to HF so the next session can resume.

Training is capped at 9 h so export + upload always fit inside Kaggle's 12 h limit. Status is written to `/kaggle/working/ohun_status.txt`.


In [ ]:
!apt-get -qq install -y build-essential cmake ninja-build > /dev/null
!git clone -q https://github.com/aladewealth/-Ohun.git ohun || (cd ohun && git pull -q)
!git clone -q --depth 1 https://github.com/OHF-voice/piper1-gpl.git || true
%cd /kaggle/working/piper1-gpl
!pip install -q scikit-build
!pip install -q -e '.[train]'
!pip install -q onnx onnxscript onnxruntime huggingface_hub
!./build_monotonic_align.sh && python3 setup.py build_ext --inplace -q
%cd /kaggle/working

In [ ]:
import os, glob, shutil, subprocess, traceback, datetime
W = '/kaggle/working'
REPO = 'aladewealth/ohun-yoruba'
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'   # torch>=2.6 defaults to weights_only=True, which breaks Lightning ckpt loading

def status(msg):
    line = f'[{datetime.datetime.utcnow():%Y-%m-%d %H:%M:%S} UTC] {msg}'
    print(line, flush=True)
    with open(f'{W}/ohun_status.txt', 'a') as f: f.write(line + '\n')

def hf_api():
    try:
        from kaggle_secrets import UserSecretsClient
        from huggingface_hub import HfApi
        return HfApi(token=UserSecretsClient().get_secret('HF_TOKEN'))
    except Exception as e:
        status(f'HF token unavailable: {e!r}'); return None

def export_onnx(ckpt):
    out = f'{W}/ohun_yo.onnx'
    r = subprocess.run(['python3', '-m', 'piper.train.export_onnx', '--checkpoint', ckpt, '--output-file', out],
                       cwd=f'{W}/piper1-gpl', capture_output=True, text=True, env=os.environ.copy())
    if r.returncode != 0 or not os.path.exists(out):
        status('EXPORT FAILED rc=%s\n%s' % (r.returncode, (r.stderr or r.stdout)[-3000:])); return None
    status(f'exported {out} ({os.path.getsize(out)/1e6:.1f} MB) from {ckpt}'); return out

def publish(include_ckpt=None):
    api = hf_api()
    if api is None: return
    for f in ['ohun_yo.onnx', 'ohun_yo.onnx.json']:
        try:
            api.upload_file(path_or_fileobj=f'{W}/{f}', path_in_repo=f, repo_id=REPO, commit_message='Ohùn Yoruba weights')
            status(f'uploaded {f} to {REPO}')
        except Exception as e: status(f'UPLOAD FAILED {f}: {e!r}')
    if include_ckpt:
        try:
            api.upload_file(path_or_fileobj=include_ckpt, path_in_repo='checkpoints/last.ckpt', repo_id=REPO, commit_message='Ohùn training checkpoint (resume point)')
            status('uploaded checkpoints/last.ckpt')
        except Exception as e: status(f'CKPT UPLOAD FAILED: {e!r}')
status('session start')

In [ ]:
!python ohun/training/prepare_yoruba.py --out /kaggle/working/data/yoruba | tee /kaggle/working/prep.log | tail -5
import re
NUM_SPK = int(re.search(r'NUM_SPEAKERS (\d+)', open('/kaggle/working/prep.log').read()).group(1))
status(f'{NUM_SPK} speakers')

In [ ]:
# Find a checkpoint to resume from: attached input first, then the HF repo.
PREV = None
cands = sorted(glob.glob('/kaggle/input/**/last.ckpt', recursive=True))
if cands:
    os.makedirs(f'{W}/resume', exist_ok=True)
    PREV = f'{W}/resume/last.ckpt'; shutil.copy(cands[0], PREV)
    j = glob.glob('/kaggle/input/**/ohun_yo.onnx.json', recursive=True)
    if j and not os.path.exists(f'{W}/ohun_yo.onnx.json'): shutil.copy(j[0], f'{W}/ohun_yo.onnx.json')
    status(f'resume from input {cands[0]}')
else:
    try:
        from huggingface_hub import hf_hub_download
        PREV = hf_hub_download(REPO, 'checkpoints/last.ckpt', local_dir=f'{W}/resume')
        try: hf_hub_download(REPO, 'ohun_yo.onnx.json', local_dir=W)
        except Exception: pass
        status('resume from HF checkpoints/last.ckpt')
    except Exception as e:
        status(f'no previous checkpoint ({e!r}); warm-starting from base')
BASE = f'{W}/base.ckpt'
if not PREV and not os.path.exists(BASE):
    !wget -q -O {BASE} 'https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/en/en_US/ljspeech/medium/lj-med_1000.ckpt'
RESUME = f'--ckpt_path {PREV}' if PREV else f'--model.warmstart_ckpt {BASE}'
print(RESUME)

In [ ]:
# Publish what we already have before training, so the API gets a voice now.
if PREV and os.path.exists(f'{W}/ohun_yo.onnx.json'):
    if export_onnx(PREV): publish()

In [ ]:
%cd /kaggle/working/piper1-gpl
!timeout -k 300 32400 python3 -m piper.train fit \
  --data.voice_name ohun_yo \
  --data.csv_path /kaggle/working/data/yoruba/metadata.csv \
  --data.audio_dir /kaggle/working/data/yoruba/wavs \
  --data.phoneme_type text \
  --model.num_speakers {NUM_SPK} \
  --model.sample_rate 22050 \
  --data.espeak_voice yo \
  --data.cache_dir /kaggle/working/cache \
  --data.config_path /kaggle/working/ohun_yo.onnx.json \
  --data.batch_size 24 \
  --trainer.devices 1 --trainer.accelerator gpu \
  --trainer.enable_progress_bar false \
  --trainer.max_epochs 1000 \
  --trainer.default_root_dir /kaggle/working/runs \
  {RESUME} 2>&1 | grep -v -E '^\s*$' | tail -n 400 > /kaggle/working/train_tail.log
%cd /kaggle/working
!tail -40 /kaggle/working/train_tail.log

In [ ]:
# Export the newest checkpoint and publish it (+ the resume checkpoint) to HF.
ck = sorted(glob.glob(f'{W}/runs/**/last.ckpt', recursive=True) or glob.glob(f'{W}/runs/**/*.ckpt', recursive=True), key=os.path.getmtime)
ck = ck[-1] if ck else PREV
status(f'final checkpoint: {ck}')
if ck and export_onnx(ck): publish(include_ckpt=ck)
status('session end')

In [ ]:
# Listen to a sample (speaker 0)
try:
    import sys; sys.path.insert(0, f'{W}/ohun')
    from server.engines import PiperPhonemeVoice
    v = PiperPhonemeVoice(f'{W}/ohun_yo.onnx', f'{W}/ohun_yo.onnx.json')
    sr, audio = v.synthesize('Ẹ kú àárọ̀. Orúkọ mi ni Ohùn, mo ń sọ èdè Yorùbá.', speaker=0)
    from IPython.display import Audio; display(Audio(audio, rate=sr))
except Exception as e:
    status(f'sample failed: {e!r}')